In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime import SamplerV2 as Sampler
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize
from qiskit_aer import StatevectorSimulator

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Dataset
n          = 1000    # Number of generated points
TEST_SIZE  = 0.2    # Fraction used for test set
SEED       = 1      # Random seed (data split & optimizer init)
N_DIM      = 3    # Features dimension

# Optimization
MAXITER    = 200     # minimizer max iterations
optimizer = COBYLA(maxiter=MAXITER) 

# Noise 
USE_NOISE = True 
NOISE_RATE = 0.02
# ──────────────────────────────────────────────────────────────────────────────
REUPLOAD_SHOTS = 1024
RC_REUPLOAD = 8
MAXITER_REUPLOAD = 200
USE_REUPLOAD = (N_DIM == 3)

print(f"USE_REUPLOAD = {USE_REUPLOAD}")


USE_REUPLOAD = True


In [3]:
if USE_NOISE:
    nm = NoiseModel()
    err1 = depolarizing_error(NOISE_RATE, 1)
    nm.add_all_qubit_quantum_error(err1, ['u', 'h', 'ry', 'rz'])
    err2 = depolarizing_error(NOISE_RATE * 5, 2)
    nm.add_all_qubit_quantum_error(err2, ['cx'])
    backend = AerSimulator(noise_model=nm, method="density_matrix")
    sampler = Sampler(backend)
else:
    sampler = Sampler(AerSimulator())

In [4]:
def generate_nsphere_data(n_samples, n_dim, radius=None, seed=None):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, (n_samples, n_dim))
    if radius is None:
        radius = np.sqrt(n_dim / 3)
    dist_sq = np.sum(X**2, axis=1)
    y = (dist_sq >= radius**2).astype(int)
    return X, y, radius

In [5]:
X, y, R = generate_nsphere_data(n_samples=n, n_dim=N_DIM, seed=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)
print(X_train.shape, y_train.shape)

(800, 3) (800,)


In [6]:
def U_su2_reupload(qc, theta_block, omega_block, x, qubit=0):
    """
    Une couche de type U(theta + omega * x) sur un seul qubit.
    Pour N_DIM=3, on injecte directement les 3 coordonnées.
    """
    qc.u(
        theta_block[0] + omega_block[0] * x[0],
        theta_block[1] + omega_block[1] * x[1],
        theta_block[2] + omega_block[2] * x[2],
        qubit
    )

def create_reupload_circuit(x, theta, omega, measure=False):
    qc = QuantumCircuit(1)
    for r in range(RC_REUPLOAD):
        qc.h(0)
        U_su2_reupload(qc, theta[r], omega[r], x, qubit=0)
    if measure:
        qc.measure_all()
    return qc

def get_reupload_probs_batch(circuits, shots=REUPLOAD_SHOTS):
    """
    - Si USE_NOISE = True : exécute les circuits mesurés sur AerSimulator bruité.
    - Sinon : utilise StatevectorSimulator pour obtenir les probabilités exactes.
    """
    probs = []

    if USE_NOISE:
        measured_circuits = []
        for qc in circuits:
            qc_m = qc.copy()
            if qc_m.num_clbits == 0:
                qc_m.measure_all()
            measured_circuits.append(qc_m)

        transpiled_circuits = transpile(measured_circuits, backend)
        result = backend.run(transpiled_circuits, shots=shots).result()

        for qc_t in transpiled_circuits:
            counts = result.get_counts(qc_t)
            p0 = counts.get('0', 0) / shots
            p1 = counts.get('1', 0) / shots
            probs.append((p0, p1))
    else:
        sim = StatevectorSimulator()
        for qc in circuits:
            sv = sim.run(qc).result().get_statevector()
            p0 = float(np.abs(sv[0])**2)
            p1 = float(np.abs(sv[1])**2)
            probs.append((p0, p1))

    return probs


In [7]:
def reupload_cost_weighted(params, X, y):
    """
    Même esprit que dans circuit.ipynb :
    - theta : paramètres "de traitement"
    - omega : poids qui multiplient les features avant réinjection
    - alphas : poids de classe pour équilibrer le coût
    """
    theta = params[:3 * RC_REUPLOAD].reshape(RC_REUPLOAD, 3)
    omega = params[3 * RC_REUPLOAD:6 * RC_REUPLOAD].reshape(RC_REUPLOAD, 3)
    alphas = params[6 * RC_REUPLOAD:]  # alpha_0, alpha_1

    circuits = [create_reupload_circuit(x, theta, omega, measure=False) for x in X]
    probs = get_reupload_probs_batch(circuits)

    total_cost = 0.0
    for i, y_target in enumerate(y):
        p0, p1 = probs[i]
        y_expected = (1.0, 0.0) if y_target == 0 else (0.0, 1.0)
        weight = alphas[y_target] ** 2
        total_cost += weight * ((p0 - y_expected[0])**2 + (p1 - y_expected[1])**2)

    return 0.5 * total_cost / len(X)

def unpack_reupload_params(params):
    theta = params[:3 * RC_REUPLOAD].reshape(RC_REUPLOAD, 3)
    omega = params[3 * RC_REUPLOAD:6 * RC_REUPLOAD].reshape(RC_REUPLOAD, 3)
    alphas = params[6 * RC_REUPLOAD:]
    return theta, omega, alphas

def optimize_reupload_parameters(X, y):
    rng = np.random.default_rng(SEED)
    init = rng.uniform(-np.pi, np.pi, size=6 * RC_REUPLOAD + 2)

    reupload_cost_history = []

    def objective(params):
        cost = reupload_cost_weighted(params, X, y)
        reupload_cost_history.append(cost)
        print(f"Iteration {len(reupload_cost_history):3d} | cost = {cost:.6f}")
        return cost

    res = minimize(
        objective,
        init,
        method="COBYLA",
        options={"maxiter": MAXITER_REUPLOAD}
    )

    theta_opt, omega_opt, alphas_opt = unpack_reupload_params(res.x)
    return res, theta_opt, omega_opt, alphas_opt, reupload_cost_history

def predict_reupload_batch(X, theta, omega):
    circuits = [create_reupload_circuit(x, theta, omega, measure=False) for x in X]
    probs = get_reupload_probs_batch(circuits)
    return np.array([0 if p0 >= p1 else 1 for p0, p1 in probs])

def evaluate_binary_predictions(y_true, y_pred, positive_label=1):
    tp = np.sum((y_pred == positive_label) & (y_true == positive_label))
    fp = np.sum((y_pred == positive_label) & (y_true != positive_label))
    fn = np.sum((y_pred != positive_label) & (y_true == positive_label))
    tn = np.sum((y_pred != positive_label) & (y_true != positive_label))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = np.mean(y_pred == y_true)

    return {
        "precision": precision,
        "accuracy": accuracy,
        "recall": recall,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

def evaluate_reupload(X, y, theta, omega, positive_label=1):
    y_pred = predict_reupload_batch(X, theta, omega)
    metrics = evaluate_binary_predictions(y, y_pred, positive_label=positive_label)
    metrics["y_pred"] = y_pred
    return metrics

In [8]:
# Study: Precision vs Number of Re-upload Layers
layers_list = [1, 2, 4, 6, 8, 10]
precisions = []
accuracies = []

for L in layers_list:
    print(f"\n--- Testing with {L} layers ---")
    global RC_REUPLOAD
    RC_REUPLOAD = L
    
    # Optimize parameters for this number of layers
    res, t_opt, o_opt, a_opt, history = optimize_reupload_parameters(X_train, y_train)
    
    # Evaluate on test set
    metrics = evaluate_reupload(X_test, y_test, t_opt, o_opt)
    precisions.append(metrics['precision'])
    accuracies.append(metrics['accuracy'])
    print(f"Layers: {L} | Precision: {metrics['precision']:.4f} | Accuracy: {metrics['accuracy']:.4f}")

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(layers_list, precisions, marker='o', label='Precision')
plt.plot(layers_list, accuracies, marker='s', linestyle='--', label='Accuracy')
plt.xlabel('Number of Re-upload Layers')
plt.ylabel('Metric Value')
plt.title('Performance vs Number of Re-upload Layers')
plt.grid(True)
plt.legend()
plt.show()



--- Testing with 1 layers ---
Iteration   1 | cost = 0.721734
Iteration   2 | cost = 0.501393
Iteration   3 | cost = 0.502194
Iteration   4 | cost = 0.736781
Iteration   5 | cost = 0.665212
Iteration   6 | cost = 0.503119
Iteration   7 | cost = 0.507889
Iteration   8 | cost = 1.057982
Iteration   9 | cost = 0.482396
Iteration  10 | cost = 0.148814
Iteration  11 | cost = 0.083633
Iteration  12 | cost = 0.053301
Iteration  13 | cost = 0.063596
Iteration  14 | cost = 0.068504
Iteration  15 | cost = 0.051341
Iteration  16 | cost = 0.399022
Iteration  17 | cost = 0.050705
Iteration  18 | cost = 0.001994
Iteration  19 | cost = 0.125801
Iteration  20 | cost = 0.007206
Iteration  21 | cost = 0.000858
Iteration  22 | cost = 0.026033
Iteration  23 | cost = 0.003453
Iteration  24 | cost = 0.027127
Iteration  25 | cost = 0.000903
Iteration  26 | cost = 0.000198
Iteration  27 | cost = 0.000216
Iteration  28 | cost = 0.011838
Iteration  29 | cost = 0.000049
Iteration  30 | cost = 0.010672
Iteration